**NOTICE**
Need to update dates within main loop.  See comments.

In [1]:
from constants import CONFIGS
from VaRBacktesting import VarBacktest
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.stats
import warnings

In [2]:
output_file = Path(CONFIGS["data_settings"]["loc_output"] + "\\" + CONFIGS["data_settings"]["output_filename"])

In [3]:
def LogLikelihood_test(break_ser, converge_rate = 0.01, alpha=0.99):

    T = len(break_ser)
    N = break_ser.sum()

    # Kupiec(1995)
    if N == T:
        LR_uc = float("inf")
    else:
        LR_uc = -2 * np.log((1-converge_rate)**(T-N)*(converge_rate**N)) + 2 * np.log((1-N/T)**(T-N)*(N/T)**N)

    # Christoffersen (1998)
    df_break = break_ser
    T00,T01,T10,T11 = 0,0,0,0
    for i in range(len(df_break)-1):
        if i == 0:
            if df_break[0] == 0:
                T00 = T00 + 1
            else:
                T01 = T01 + 1
        if df_break[i] == 0:
            if df_break[i+1] == 0:
                T00 = T00 + 1
            else:
                T01 = T01 + 1
        elif df_break[i] == 1:
            if df_break[i+1] == 0:
                T10 = T10 + 1
            else:
                T11 = T11 + 1   
    
    if N == T:
        LR_ind = float("inf")
    elif N == 0:
        LR_ind = 0
    else:
        if T00 == 0:
            pi0 = 1
        else:
            pi0 = T01 / (T - N)
        pi  = N / T
        pi1 = T11 / N
        LR_ind = -2 * np.log((1-pi)**(T00+T10) * pi**(T01+T11)) + 2 * np.log((1-pi0)**T00 * pi0**T01 * (1-pi1)**T10 * pi1**T11)
  
    LR_cc = LR_uc + LR_ind
    
    chi2_1 = scipy.stats.chi2.ppf(alpha,1)
    chi2_2 = scipy.stats.chi2.ppf(alpha,2)

    result = {
        'Exceptions': break_ser.sum(),
        'LR_uc':round(LR_uc,3), 
        'LR_ind':round(LR_ind,3), 
        'LR_cc':round(LR_cc,3), 
        f'chi2(1) {int(alpha*100)}%':round(chi2_1,3), 
        f'chi2(2) {int(alpha*100)}%':round(chi2_2,3),
        "Independence": "RejectNull" if LR_ind > chi2_1 else None,
        "alpha": round(alpha * 100, 1),
        "converge_rate": round(converge_rate * 100,1)}

    return result

**Step 2:** Iterate through each volatility factor for backtesting

In [4]:
data = pd.read_excel(f"{output_file}.xlsx", dtype={"Weights": str}) 

#----------------------Step 1: unpivot VaR values-------------------------------#
df = pd.melt(data, 
              id_vars = ["Volatility_Factor_ID", "Tickers", "Weights", "SampleStart", "SampleEnd", "BootstrapStart", "BootstrapEnd", "ForecastStart", 
                        "ForecastEnd", "StartingValueDate", "ShockType", "VaRp", "Method", "Horizon1", "Horizon2", "NDays", "StartingValueQ", "ConvergencyCode",
                        "Simulations", "MeanModel", "VolatilityModel"],
              value_vars = ["VaR_50", "VaR_75", "VaR_95", "VaR_99"],
              var_name = "VaRQ",
              value_name = "VaR")

#------------------------Step 2: create lists for iteration-------------------------#
volatility_factor_id = df["Volatility_Factor_ID"].to_list()
tickers = df["Tickers"].str.split(";").to_list()
weightsTEMP = df["Weights"].str.split(";").to_list()
weights = [[float(weight) for weight in x] for x in weightsTEMP]
sample_start = pd.to_datetime(df["SampleStart"]).dt.strftime("%Y-%m-%d").to_list() 
sample_end = pd.to_datetime(df["SampleEnd"]).dt.strftime("%Y-%m-%d").to_list() 
forecast_start = pd.to_datetime(df["ForecastStart"]).dt.strftime("%Y-%m-%d").to_list()
forecast_end = pd.to_datetime(df["ForecastEnd"]).dt.strftime("%Y-%m-%d").to_list()
bootstrap_start = pd.to_datetime(df["BootstrapStart"]).dt.strftime("%Y-%m-%d").to_list()
bootstrap_end = pd.to_datetime(df["BootstrapEnd"]).dt.strftime("%Y-%m-%d").to_list()
starting_value_date = pd.to_datetime(df["StartingValueDate"]).dt.strftime("%Y-%m-%d").to_list()
shock_type = df["ShockType"].to_list()
var_p = df["VaRp"].to_list()
method = df["Method"].to_list()
horizon1 = df["Horizon1"].to_list()
horizon2 = df["Horizon2"].to_list()
n_liquidation = df["NDays"].to_list()
starting_value_quantile = df["StartingValueQ"].to_list()
varq = (df["VaRQ"].str[-2:] + "Q").to_list()
var = df["VaR"].to_list()


#--------------------Step 3: iterate through list for backtesting--------------------#
bt_frames = [] #list to hold backtesting dataframes
ind_dicts = [] #dicts for independence tests
for i in range(len(tickers)):        
    #------Instantiate instance of VaRBacktest; Initalize df to persist computed properties (start with index = price_matrix include valid dates only)-----#
    bt = VarBacktest(tickers = tickers[i], n_liquidation = n_liquidation[i], weights = weights[i], 
                start_date = forecast_start[i], end_date = forecast_end[i], VaR = var[i], VaR_is_log = False, forward_shift=True) #instantiate backtest instnace
    df = pd.DataFrame(data = None, index = bt.price_matrix[forecast_start[i]:forecast_end[i]].index) #initalize df to persist bt properties.  Start with price matrix
    if df.shape[0] == 0:
        df = pd.DataFrame(data = None, index = pd.date_range(start = forecast_start[i], end = forecast_end[i])) #<where no prices exist ()
    elif df.index.max() == pd.to_datetime("2024-12-31"):#<-------------------------------------------------------------------------------------------------------------------------UPDATE
            df = pd.concat([df, pd.DataFrame(data = None, index = pd.date_range(start = "2025-01-01", end = "2025-03-31"))]) #<----------------------------------------------------UPDATE

    #-------Independence Test------#
    warnings.filterwarnings('ignore') #turn-off warnings to supress div by 0; turned back on after function called
    ind_test = LogLikelihood_test(bt.break_series)
    ind_test.update({
        "Volatility_Factor_ID": bt.volatility_factor_id,
        "ForecastStart": pd.to_datetime(bt.start_date).strftime("%Y-%m-%d"),
        "ForecastEnd" : pd.to_datetime(bt.end_date).strftime("%Y-%m-%d")})
    ind_dicts.append(ind_test)
    warnings.filterwarnings('default')

    #-------Backtest Detail------#
    df["Date"] = df.index
    df["Volatility_Factor_ID"] = bt.volatility_factor_id
    df["ForecastStart"] = pd.to_datetime(bt.start_date).strftime("%Y-%m-%d")
    df["ForecastEnd"] = pd.to_datetime(bt.end_date).strftime("%Y-%m-%d")
    df["Method"] = method[i]
    df["NDays"] = bt.n_liquidation
    df["VaRQ"] = varq[i] 
    df["VaR_Pct"] = bt.var_pct  
    df["Price"] = bt.wtd_price
    df["Pct_Change_Actual"] = bt.nday_price_return
    df["Breaks"] = bt.break_series
    df["ShortfallPct"] = bt.shortfall_series
    df["RejectNullIndependence"] = np.where(ind_test.get("Independence") == "RejectNull", 1, 0)
    bt_frames.append(df)

#------Save Results-------#
df = pd.concat(bt_frames)
df = df.reset_index(drop=True)
df.to_parquet(f"{output_file.parent}\\backtesting_detail_{output_file.name}.parquet", compression = "Snappy")

dfi = pd.DataFrame.from_dict(ind_dicts)
dfi.to_excel(f"{output_file.parent}\\independence_tests_{output_file.name}.xlsx", index=False)